# Derivatives Pricing Analytics
---
> **Bathaix Philippe-Emmanuel Yao**

This notebook consolidates three derivative pricing projects into a unified, structured library:

| Module | Instruments | Methods |
|---|---|---|
| **I. Shared Infrastructure** | Yield curve, vol skew, swaption | Cubic spline bootstrapping, Bachelier |
| **II. CMS Derivatives** | CMS forwards, caplets/floorlets | Carr-Madan replication, Linear Mean-Reversion TSR |
| **III. CMS Spread Options** | Spread caplets/floorlets | Bivariate Gaussian copula, Breeden-Litzenberger, double Simpson |
| **IV. American Options** | American calls/puts | Binomial tree, BAW approximation, Longstaff-Schwartz LSMC |


---
## I. Shared Infrastructure

All modules below depend on a common set of building blocks: a zero-coupon curve, a normal volatility skew,
and the Bachelier (Normal model) swaption pricer. Centralising these avoids duplication and ensures
consistency across the three pricing engines.


### I.1 — Global Imports

In [ ]:
import numpy as np
import math
import warnings
from scipy.interpolate import interp1d
from scipy.stats import norm
from scipy import optimize
from scipy.integrate import quad
import matplotlib.pyplot as plt

warnings.simplefilter('ignore', np.RankWarning)
np.random.seed(0)


### I.2 — Zero-Coupon Curve

Bootstraps a continuously-compounded ZC curve from market tenors and rates.
Cubic spline interpolation is used within the pillar range; extrapolation is flat at the wings.


In [ ]:
class zc_curve:
    """
    Zero-Coupon Rates Curve.
    Stores market pillars and provides discount factor B(0, T) = exp(-r(T) * T).
    """
    def __init__(self, maturities, zc_rates):
        # Pillar maturities and continuously-compounded ZC rates
        self.maturities = maturities
        self.zc_rates = zc_rates
        # Cubic spline; extrapolation enabled for tenors outside the pillar range
        self.zc_rates_interp = interp1d(
            maturities, zc_rates, kind='cubic', fill_value="extrapolate"
        )

    def df(self, T):
        """Discount factor B(0, T)."""
        return np.exp(-self.zc_rates_interp(T) * T)


### I.3 — Normal Volatility Skew

Represents a single-expiry normal (Bachelier) vol skew.
Cubic interpolation is used within the quoted strike range; linear extrapolation beyond it.


In [ ]:
class vol_skew:
    """
    Normal (Bachelier) implied vol skew for a single expiry.
    Interpolation: cubic inside market strikes, linear outside.
    """
    def __init__(self, strikes, vol_data):
        self.strikes = strikes
        self.vol_data = vol_data

    def normal_vol(self, strike):
        """Returns the interpolated/extrapolated normal vol at the given strike."""
        if strike < self.strikes[0] or strike > self.strikes[-1]:
            return float(interp1d(
                self.strikes, self.vol_data, kind='linear', fill_value="extrapolate"
            )(strike))
        return float(interp1d(
            self.strikes, self.vol_data, kind='cubic', fill_value="extrapolate"
        )(strike))


### I.4 — Bachelier (Normal Model) Swaption

Implements a European payer/receiver swaption under the Bachelier (normal) model.
Provides both the present value (`market_price`) and the forward price (`market_fwd_price`)
stranded at the fixing date, used by the Breeden-Litzenberger PDF extraction and the
Carr-Madan replication engine.


In [ ]:
class swaption:
    """
    European swaption priced under the Normal (Bachelier) model.

    Conventions:
      - Fixed leg pays annually; start_time = expiry + 2 business days.
      - level = sum of DCF_i * B(0, T_i) over fixed leg payment dates.
      - forward swap rate S(0) = (B(0, T_start) - B(0, T_end)) / level.
    """
    def __init__(self, payer_receiver, expiry, tenor, strike, notional=1.0):
        self.payer_receiver = payer_receiver
        self.expiry = expiry
        self.strike = strike
        self.notional = notional
        self.start_time = expiry + 2.0 / 365.0
        # Annual fixed leg payment dates
        self.pay_times = np.arange(self.start_time + 1, expiry + tenor + 1)
        self.year_fractions = np.ones(len(self.pay_times))

    def set_market_data(self, df, normal_vol):
        """Cache discount factors and store the normal vol for a given strike."""
        self.df_exp_time   = df(self.expiry)
        self.df_start_time = df(self.start_time)
        self.df_pay_times  = df(self.pay_times)
        self.normal_vol    = normal_vol

    def level(self):
        """Annuity: sum_i DCF_i * B(0, T_i)."""
        return np.sum(self.df_pay_times * self.year_fractions)

    def forward(self):
        """Forward swap rate S(0) = (B(0,T_start) - B(0,T_end)) / level."""
        return (self.df_start_time - self.df_pay_times[-1]) / self.level()

    def pv_underlying(self):
        """PV of the underlying (unfunded) swap."""
        phi = 1.0 if self.payer_receiver.upper() == 'PAYER' else -1.0
        return phi * self.notional * self.level() * (self.forward() - self.strike)

    def market_price(self):
        """Bachelier swaption present value (level-discounted)."""
        phi = 1 if self.payer_receiver.upper() == 'PAYER' else -1
        lvl, fwd = self.level(), self.forward()
        if self.expiry == 0 or self.normal_vol == 0:
            return max(phi * (fwd - self.strike), 0.0)
        sqrt_V2T = self.normal_vol * math.sqrt(self.expiry)
        d = (fwd - self.strike) / sqrt_V2T
        return self.notional * lvl * sqrt_V2T * (phi * d * norm.cdf(phi * d) + norm.pdf(d))

    def market_fwd_price(self):
        """Bachelier swaption forward price (not discounted by the level)."""
        phi = 1 if self.payer_receiver.upper() == 'PAYER' else -1
        fwd = self.forward()
        if self.expiry == 0 or self.normal_vol == 0:
            return max(phi * (fwd - self.strike), 0.0)
        sqrt_V2T = self.normal_vol * math.sqrt(self.expiry)
        d = (fwd - self.strike) / sqrt_V2T
        return self.notional * sqrt_V2T * (phi * d * norm.cdf(phi * d) + norm.pdf(d))


---
## II. CMS Derivatives Pricing

CMS (Constant Maturity Swap) products deliver a payoff $f(S(T_f, T_0, T_1))$ at $T_p$,
where $S$ is a swap rate with fixed maturity $T_1 - T_0$ observed at $T_f$.

The present value under the $T_p$-forward measure is:

$$PV = B(0, T_p) \times E^{Q^{T_p}}\!\left[f\left(S(T_f)\right)\right]$$

We change measure to the level (annuity) measure $Q^{\text{LVL}}$, under which $S(t)$ is a martingale,
and approximate $B(T_f, T_p) / \text{LVL}(T_f) \approx g(S(T_f)) := a \cdot S(T_f) + b$
using the **Linear Mean-Reversion TSR** model calibrated to the Hull-White one-factor framework.

The resulting expectations are evaluated by **Carr-Madan static replication** via OTM swaptions.


### II.1 — Linear Mean-Reversion TSR Model

The slope $a$ and intercept $b$ of the linear TSR approximation $g(s) = as + b$ are derived
from Hull-White one-factor zero-coupon bond sensitivities evaluated at the current forward swap rate.

$$a = \frac{B(0, T_p)(\gamma - \beta(T_f, T_p))}{B(0, T_N)\beta(T_f, T_N) + \text{LVL}(0)\cdot S(0)\cdot\gamma}, \qquad b = \frac{B(0, T_p)}{\text{LVL}(0)} - a \cdot S(0)$$

where $\gamma = \frac{\sum_i DCF_i \cdot B(0, T^p_i)\cdot\beta(T_f, T^p_i)}{\text{LVL}(0)}$
and $\beta(t, T) = \frac{1 - e^{-\kappa(T-t)}}{\kappa}$ is the HW-1F bond-price sensitivity.


In [ ]:
class tsr_model:
    """
    Terminal Swap Rate model — Linear Mean-Reversion parameterisation.

    Provides TSR coefficients (a, b) such that:
        B(T_f, T_p) / LVL(T_f) ≈ a * S(T_f) + b

    Calibrated to the Hull-White one-factor model with mean-reversion kappa.
    """
    def __init__(self, df, mean_reversion=0.02):
        self.df = df
        self.mean_reversion = mean_reversion  # kappa

    def beta(self, t, T):
        """HW-1F bond-price sensitivity: beta(t,T) = (1 - exp(-kappa*(T-t))) / kappa."""
        return (1 - np.exp(-self.mean_reversion * (T - t))) / self.mean_reversion

    def tsr_coeffs(self, expiry, tenor, pay_date):
        """
        Returns {'a': slope, 'b': intercept} of the linear TSR function.
        Returns None if expiry == 0 (no convexity adjustment required).
        """
        if expiry == 0:
            print("The input expiry is 0: TSR model not applied.")
            return None

        start_time = expiry + 2.0 / 365.0
        pay_times = np.arange(start_time + 1, expiry + tenor + 1)   # annual fixed-leg dates
        year_fractions = np.ones(len(pay_times))

        df_start  = self.df(start_time)
        df_pays   = self.df(pay_times)
        level     = np.sum(df_pays * year_fractions)
        swap_fwd  = (df_start - df_pays[-1]) / level
        gamma     = np.sum(df_pays * year_fractions * self.beta(expiry, pay_times)) / level

        num = self.df(pay_date) * (gamma - self.beta(expiry, pay_date))
        den = df_pays[-1] * self.beta(expiry, pay_times[-1]) + level * swap_fwd * gamma
        a = num / den
        b = self.df(pay_date) / level - a * swap_fwd

        return {"a": a, "b": b}


### II.2 — Carr-Madan Replication Engine

For $h(s) = f(s) \cdot g(s)$, the Carr-Madan decomposition reads:

$$E^{Q^{\text{LVL}}}[h(S)] = h(S_0) + h'(S_0)(S(T_f)-S_0) + \int_{-\infty}^{S_0} h''(k)\,(k-S)^+\,dk + \int_{S_0}^{+\infty} h''(k)\,(S-k)^+\,dk$$

**CMS Forward** ($f(s)=s$): $h(s) = as^2 + bs$, $h''(s) = 2a$
**CMS Caplet** ($f(s)=(s-K)^+$): adjusted replication with a boundary term at the strike $K$
**CMS Floorlet**: derived from CMS caplet via put-call parity on the CMS forward


In [ ]:
class replication_method:
    """
    Carr-Madan static replication of CMS forwards, caplets, and floorlets
    via a portfolio of OTM swaptions.
    """
    def __init__(self, tsr_model):
        self.tsr_model = tsr_model

    def replication_price(self, payoff_type, expiry, tenor, pay_date,
                          vol_skew_fn, strike=0.0, n_stdev=5.0):
        """
        Parameters
        ----------
        payoff_type : str  — 'Forward', 'Caplet', or 'Floorlet'
        expiry      : float — option expiry (years)
        tenor       : float — swap tenor (years)
        pay_date    : float — CMS payment date (years)
        vol_skew_fn : callable — vol_skew.normal_vol or equivalent
        strike      : float — option strike (only used for Caplet/Floorlet)
        n_stdev     : float — integration truncation (in normal-vol standard deviations)

        Returns
        -------
        float — present value of the CMS instrument
        """
        df = self.tsr_model.df

        # Dates and annuity
        start_time    = expiry + 2.0 / 365.0
        pay_times     = np.arange(start_time + 1, expiry + tenor + 1)
        year_frac     = np.ones(len(pay_times))
        df_start      = df(start_time)
        df_pays       = df(pay_times)
        df_pay_date   = df(pay_date)
        level         = np.sum(df_pays * year_frac)
        swap_fwd      = (df_start - df_pays[-1]) / level

        # Boundary case: expiry already reached
        if expiry == 0:
            payoff_type = payoff_type.upper()
            if payoff_type == "FORWARD":
                return df_pay_date * swap_fwd
            elif payoff_type == "CAPLET":
                return df_pay_date * max(swap_fwd - strike, 0.0)
            elif payoff_type == "FLOORLET":
                return df_pay_date * max(strike - swap_fwd, 0.0)
            else:
                print("Undefined payoff type. Options: Forward, Caplet, Floorlet.")
                return None

        # TSR coefficients
        tsr = self.tsr_model.tsr_coeffs(expiry, tenor, pay_date)
        a, b = tsr["a"], tsr["b"]

        # Reusable swaption objects (mutate strike/vol inside lambda to avoid re-instantiation)
        swopt_pay = swaption("Payer",    expiry, tenor, swap_fwd)
        swopt_rec = swaption("Receiver", expiry, tenor, swap_fwd)
        swopt_pay.set_market_data(df, vol_skew_fn(swap_fwd))
        swopt_rec.set_market_data(df, vol_skew_fn(swap_fwd))

        def swopt_fwd_price(direction, k):
            """OTM swaption forward price at strike k."""
            if direction.upper() == "PAYER":
                swopt_pay.strike     = k
                swopt_pay.normal_vol = vol_skew_fn(k)
                return swopt_pay.market_fwd_price()
            else:
                swopt_rec.strike     = k
                swopt_rec.normal_vol = vol_skew_fn(k)
                return swopt_rec.market_fwd_price()

        # Integration bounds (n_stdev * normal std of swap rate)
        atm_vol     = vol_skew_fn(swap_fwd)
        upper_bound = swap_fwd + n_stdev * atm_vol * math.sqrt(expiry)
        lower_bound = swap_fwd - n_stdev * atm_vol * math.sqrt(expiry)

        # ── CMS FORWARD ──────────────────────────────────────────────────────
        # h(s) = a*s^2 + b*s  =>  h''(s) = 2a
        payoff_sd   = lambda k: 2 * a
        otm_put_fn  = lambda k: payoff_sd(k) * swopt_fwd_price("Receiver", k)
        otm_call_fn = lambda k: payoff_sd(k) * swopt_fwd_price("Payer",    k)

        h_fwd    = a * swap_fwd**2 + b * swap_fwd
        rep_fwd  = h_fwd + quad(otm_put_fn, lower_bound, swap_fwd)[0]                          + quad(otm_call_fn, swap_fwd, upper_bound)[0]
        cms_fwd  = level * rep_fwd / df_pay_date

        if payoff_type.upper() == "FORWARD":
            print({
                "Swap Fwd (%)": round(100 * swap_fwd, 4),
                "CMS Fwd (%)":  round(100 * cms_fwd,  4),
                "Conv. Adj. (%)": round(100 * (cms_fwd - swap_fwd), 4)
            })
            return cms_fwd

        # ── CMS CAPLET / FLOORLET ─────────────────────────────────────────────
        # h(s) = (a*s + b)*(s - K)  =>  h''(s) = 2a for s > K
        elif payoff_type.upper() in ("CAPLET", "FLOORLET"):
            g       = lambda s: a * s + b
            rep_cap = g(swap_fwd) * max(swap_fwd - strike, 0.0)                     + g(strike) * swopt_fwd_price(
                        "Receiver" if strike < swap_fwd else "Payer", strike)                     + quad(otm_put_fn,  min(strike, swap_fwd), swap_fwd)[0]                     + quad(otm_call_fn, max(strike, swap_fwd), upper_bound)[0]

            caplet_pv = level * rep_cap
            if payoff_type.upper() == "CAPLET":
                return caplet_pv
            else:
                # Put-call parity: Floorlet = Caplet - B(0,Tp)*(CMS_fwd - K)
                return caplet_pv - df_pay_date * (cms_fwd - strike)
        else:
            print("Undefined payoff type. Options: Forward, Caplet, Floorlet.")
            return None


### II.3 — Numerical Application: 10Y EUR CMS (5Y Expiry, Paid 6Y)

In [ ]:
# ── Market data: EURIBOR 6M curve as of 1 February 2024 ──────────────────────
maturities_eur = np.array([0.5, 1, 2, 5, 6, 8, 10, 15, 20, 30])
zc_rates_eur   = np.array([3.84, 3.41, 2.84, 2.48, 2.47, 2.49, 2.52, 2.60, 2.53, 2.28]) / 100
yc_eur = zc_curve(maturities_eur, zc_rates_eur)

# ── Swaption vol skew: 5Y x 10Y normal vols ──────────────────────────────────
expiry_cms = 5.0
tenor_cms  = 10.0
pay_cms    = 6.0

strikes_5y10y    = np.array([1.18, 1.68, 2.18, 2.68, 3.68, 4.68, 5.18]) / 100
normal_vols_5y10y = np.array([84.70, 83.81, 83.76, 84.74, 89.82, 98.07, 102.91]) / 1e4
vol_skew_5y10y    = lambda k: vol_skew(strikes_5y10y, normal_vols_5y10y).normal_vol(k)

# ── TSR model (kappa = 1.5%) & replication engine ────────────────────────────
tsr_cms  = tsr_model(yc_eur.df, mean_reversion=0.015)
rep      = replication_method(tsr_cms)

# ── CMS forward & convexity adjustment ───────────────────────────────────────
print("5Yx10Y CMS Forward paid in 6Y:")
print("─" * 40)
cms_fwd_val = rep.replication_price("Forward", expiry_cms, tenor_cms, pay_cms, vol_skew_5y10y)


In [ ]:
# ── ATM CMS caplet & floorlet ─────────────────────────────────────────────────
print("\nATM 5Yx10Y CMS Caplet & Floorlet (paid 6Y):")
print("─" * 50)
caplet_atm  = rep.replication_price("Caplet",   expiry_cms, tenor_cms, pay_cms, vol_skew_5y10y, cms_fwd_val)
floorlet_atm = rep.replication_price("Floorlet", expiry_cms, tenor_cms, pay_cms, vol_skew_5y10y, cms_fwd_val)
print(f"Caplet price  (bps): {round(1e4 * caplet_atm,  4)}")
print(f"Floorlet price (bps): {round(1e4 * floorlet_atm, 4)}")

# ── OTM CMS caplet & floorlet at K = 2.5% ────────────────────────────────────
print("\n5Yx10Y CMS Caplet & Floorlet at strike = 2.50% (paid 6Y):")
print("─" * 60)
K_otm = 0.025
caplet_otm   = rep.replication_price("Caplet",   expiry_cms, tenor_cms, pay_cms, vol_skew_5y10y, K_otm)
floorlet_otm = rep.replication_price("Floorlet", expiry_cms, tenor_cms, pay_cms, vol_skew_5y10y, K_otm)
print(f"Caplet price  (bps): {round(1e4 * caplet_otm,  4)}")
print(f"Floorlet price (bps): {round(1e4 * floorlet_otm, 4)}")


---
## III. CMS Spread Options Pricing

A CMS spread caplet with fixing date $T_f$, payment date $T_p$, tenors $(T_1, T_2)$, and strike $K$ pays:

$$\max\left(S^1_{T_f} - S^2_{T_f} - K,\; 0\right) \quad \text{at } T_p$$

The present value under the $T_p$-forward measure factorises as a double integral:

$$PV = B(0, T_p) \iint \max(s_1 - s_2 - K,\, 0)\; c(F_1(s_1), F_2(s_2))\; f_1(s_1)\; f_2(s_2)\; ds_1\, ds_2$$

where $c$ is the bivariate **Gaussian copula** density, and $(f_i, F_i)$ are the PDF/CDF of
$S^i_{T_f}$ under the forward measure $Q^{T_p}$, obtained via:

1. **Breeden-Litzenberger** finite-difference extraction of the level-measure PDF from swaption prices
2. **Radon-Nikodym** change of measure using the Linear Mean-Reversion TSR function $g(s) = as + b$

The double integral is evaluated by **double Simpson quadrature** on a fixed grid.


### III.1 — Mathematical Utilities (Gaussian Copula + Simpson Quadrature)

In [ ]:
class math_utils:
    """
    Numerical tools used by the copula pricing engine:
      - Bivariate Gaussian copula density
      - Single and double Simpson integration
    """

    def gaussian_copula(self, u1, u2, rho):
        """
        Bivariate Gaussian copula density c(u1, u2; rho).
        
        c(u1, u2; rho) = exp(num/den) / sqrt(1 - rho^2)
        where num = 2*rho*z1*z2 - rho^2*(z1^2 + z2^2),  den = 2*(1-rho^2)
        and z_i = Phi^{-1}(u_i).
        """
        if u1 == 0 or u2 == 0:
            return 0.0
        z1, z2 = norm.ppf(u1), norm.ppf(u2)
        num = 2 * rho * z1 * z2 - rho**2 * (z1**2 + z2**2)
        den = 2 * (1 - rho**2)
        return math.exp(num / den) / math.sqrt(1 - rho**2)

    def simpson_integral(self, f, h, lx, ux):
        """
        Composite Simpson's rule for a single integral on [lx, ux] with step h.
        """
        nx = round((ux - lx) / h + 1)
        result = 0.0
        for i in range(nx):
            xi = lx + i * h
            if i == 0 or i == nx - 1:
                result += f(xi)
            elif i % 2 == 0:
                result += 2 * f(xi)
            else:
                result += 4 * f(xi)
        return result * h / 3

    def simpson_double_integral(self, f, h, k, lx, ux, ly, uy):
        """
        Composite double Simpson's rule on [lx, ux] x [ly, uy] with steps (h, k).
        Inner y-integral is computed first for each x node, then the outer x-integral.
        """
        nx = round((ux - lx) / h + 1)
        ny = round((uy - ly) / k + 1)

        # Inner integrals for each x node
        ax = np.zeros(nx)
        for i in range(nx):
            xi = lx + i * h
            for j in range(ny):
                yj = ly + j * k
                w = 1 if (j == 0 or j == ny - 1) else (2 if j % 2 == 0 else 4)
                ax[i] += w * f(xi, yj)
            ax[i] *= k / 3

        # Outer integral
        result = 0.0
        for i in range(nx):
            w = 1 if (i == 0 or i == nx - 1) else (2 if i % 2 == 0 else 4)
            result += w * ax[i]
        return result * h / 3


### III.2 — Copula Pricing Engine

In [ ]:
class copula_method:
    """
    Bivariate Gaussian copula pricer for CMS spread caplets and floorlets.

    Workflow:
      1. Extract the level-measure PDF via Breeden-Litzenberger (finite differences on Bachelier prices)
      2. Change to the T_p-forward measure via the Radon-Nikodym TSR weight g(s) = a*s + b
      3. Compute the CDF by Simpson integration of the forward-measure PDF
      4. Price via double Simpson integration of the copula-weighted payoff
    """
    def __init__(self, tsr):
        self.tsr_model = tsr
        self.df        = tsr.df
        self.math      = math_utils()

    def set_data(self, expiry, cms_tenor_1, cms_tenor_2, pay_date,
                 vol_skew_fn_1, vol_skew_fn_2, correl):
        """Configure the pricer for a specific CMS spread option."""
        self.expiry      = expiry
        self.cms_tenor_1 = cms_tenor_1
        self.cms_tenor_2 = cms_tenor_2
        self.pay_date    = pay_date
        self.vol_skew_1  = vol_skew_fn_1
        self.vol_skew_2  = vol_skew_fn_2
        self.correl      = correl
        self._set_params()

    def _set_params(self):
        """Pre-compute levels and TSR coefficients (called once per set_data)."""
        self.df_pay_date = self.df(self.pay_date)
        self.start_time  = self.expiry + 2.0 / 365.0

        def _level(tenor):
            pts = np.arange(self.start_time + 1, self.expiry + tenor + 1)
            return np.sum(self.df(pts))

        self.level_1      = _level(self.cms_tenor_1)
        self.level_2      = _level(self.cms_tenor_2)
        self.tsr_coeffs_1 = self.tsr_model.tsr_coeffs(self.expiry, self.cms_tenor_1, self.pay_date)
        self.tsr_coeffs_2 = self.tsr_model.tsr_coeffs(self.expiry, self.cms_tenor_2, self.pay_date)

    def pdf_lvl_measure(self, k, cms_tenor, vol_skew_fn):
        """
        Breeden-Litzenberger PDF under the level measure.
        f(k) = d^2 Price_swaption / dk^2  approximated by 2nd-order central finite differences.
        OTM convention: payer if k >= forward, receiver if k < forward.
        """
        eps  = 1e-4  # 1 bps shift
        swop = swaption("Payer", self.expiry, cms_tenor, k)
        swop.set_market_data(self.df, vol_skew_fn(k))
        if k < swop.forward():
            swop.payer_receiver = "Receiver"

        def fwd_price(strike):
            swop.strike     = strike
            swop.normal_vol = vol_skew_fn(strike)
            return swop.market_fwd_price()

        return (fwd_price(k + eps) + fwd_price(k - eps) - 2 * fwd_price(k)) / eps**2

    def pdf_fwd_measure(self, k, cms_tenor, level, vol_skew_fn, tsr_coeffs):
        """
        Forward-measure PDF via Radon-Nikodym:
        f^{Q^Tp}(k) = (level / B(0,Tp)) * g(k) * f^{LVL}(k)
        where g(k) = a*k + b (linear TSR approximation).
        """
        g = tsr_coeffs["a"] * k + tsr_coeffs["b"]
        return (level / self.df(self.pay_date)) * g * self.pdf_lvl_measure(k, cms_tenor, vol_skew_fn)

    def cdf_fwd_measure(self, k, cms_tenor, level, vol_skew_fn, tsr_coeffs):
        """
        Forward-measure CDF: Simpson integration of the forward-measure PDF from -4% to k.
        Step size 50 bps; lower bound chosen to safely capture the left tail.
        """
        return self.math.simpson_integral(
            lambda s: self.pdf_fwd_measure(s, cms_tenor, level, vol_skew_fn, tsr_coeffs),
            h=0.005, lx=-0.04, ux=k
        )

    def price(self, flavor, strike):
        """
        CMS spread option price via bivariate Gaussian copula + double Simpson quadrature.

        Parameters
        ----------
        flavor : str   — 'Caplet' or 'Floorlet'
        strike : float — spread strike

        Returns
        -------
        float — present value (unit notional)
        """
        flavor = flavor.upper()
        if flavor == "CAPLET":
            phi = 1.0
        elif flavor == "FLOORLET":
            phi = -1.0
        else:
            print("Undefined flavor. Options: Caplet, Floorlet.")
            return None

        # Boundary case: expiry already reached
        if self.expiry == 0:
            def _fwd(tenor):
                sw = swaption("Payer", 0, tenor, strike)
                sw.set_market_data(self.df, 0)
                return sw.forward()
            return self.df(self.pay_date) * max(phi * (_fwd(self.cms_tenor_1) - _fwd(self.cms_tenor_2) - strike), 0)

        # Integrand: payoff * joint density (copula-weighted)
        def integrand(s1, s2):
            payoff = max(phi * (s1 - s2 - strike), 0.0)
            if payoff == 0.0:
                return 0.0
            f1  = self.pdf_fwd_measure(s1, self.cms_tenor_1, self.level_1, self.vol_skew_1, self.tsr_coeffs_1)
            f2  = self.pdf_fwd_measure(s2, self.cms_tenor_2, self.level_2, self.vol_skew_2, self.tsr_coeffs_2)
            u1  = self.cdf_fwd_measure(s1, self.cms_tenor_1, self.level_1, self.vol_skew_1, self.tsr_coeffs_1)
            u2  = self.cdf_fwd_measure(s2, self.cms_tenor_2, self.level_2, self.vol_skew_2, self.tsr_coeffs_2)
            cop = self.math.gaussian_copula(u1, u2, self.correl)
            return payoff * f1 * f2 * cop

        # Fixed grid: step 50 bps, range [-4%, +8%]
        pv = self.math.simpson_double_integral(
            integrand, h=0.005, k=0.005, lx=-0.04, ux=0.08, ly=-0.04, uy=0.08
        )
        return self.df(self.pay_date) * pv


### III.3 — Numerical Application: 10Y–2Y EUR CMS Spread Options (5Y Expiry)

In [ ]:
# ── Market data (same EURIBOR 6M curve as Section II) ────────────────────────
expiry_sp = 5.0
pay_sp    = expiry_sp + 14.0 / 365.0   # paid 2 weeks after fixing

# 5Y x 10Y vol skew (reuse from Section II)
tenor_10y       = 10.0
vol_skew_fn_10y = vol_skew_5y10y       # defined in Section II

# 5Y x 2Y vol skew
tenor_2y           = 2.0
strikes_5y2y       = np.array([0.99, 1.49, 1.99, 2.49, 3.49, 4.49, 4.99]) / 100
normal_vols_5y2y   = np.array([88.44, 88.72, 89.62, 91.21, 96.39, 103.72, 107.93]) / 1e4
vol_skew_fn_2y     = lambda k: vol_skew(strikes_5y2y, normal_vols_5y2y).normal_vol(k)

# Historical correlation between CMS10Y and CMS2Y at the 5Y horizon
corr_5y = 0.81

# ── TSR model & copula engine ─────────────────────────────────────────────────
tsr_sp = tsr_model(yc_eur.df)          # default kappa = 2%
cop    = copula_method(tsr_sp)
cop.set_data(expiry_sp, tenor_10y, tenor_2y, pay_sp,
             vol_skew_fn_10y, vol_skew_fn_2y, corr_5y)


In [ ]:
# ── Vol skew visualisation ────────────────────────────────────────────────────
ks = np.linspace(-0.02, 0.08, 31)
plt.figure(figsize=(9, 4))
plt.scatter(ks, [vol_skew_fn_2y(k)  for k in ks], color="red",  label="Normal Vol 5Yx2Y")
plt.scatter(ks, [vol_skew_fn_10y(k) for k in ks], color="blue", label="Normal Vol 5Yx10Y")
plt.xlabel("Strike"); plt.ylabel("Normal Vol"); plt.title("Swaption Vol Skews (5Y Expiry)")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
# ── Level-measure PDFs ────────────────────────────────────────────────────────
ks_pdf = np.linspace(-0.05, 0.10, 31)
pdf_2y  = [cop.pdf_lvl_measure(k, tenor_2y,  vol_skew_fn_2y)  for k in ks_pdf]
pdf_10y = [cop.pdf_lvl_measure(k, tenor_10y, vol_skew_fn_10y) for k in ks_pdf]

plt.figure(figsize=(9, 4))
plt.scatter(ks_pdf, pdf_2y,  color="red",  label="LVL PDF 5Yx2Y")
plt.scatter(ks_pdf, pdf_10y, color="blue", label="LVL PDF 5Yx10Y")
plt.xlabel("Swap Rate"); plt.ylabel("Density"); plt.title("Breeden-Litzenberger PDFs under Level Measure")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


In [ ]:
# ── CMS Spread Option prices ──────────────────────────────────────────────────
print("10Y–2Y CMS Spread Options  |  Expiry = 5Y  |  Paid = 5Y + 14D")
print("─" * 65)
strikes_sp = np.linspace(0.002, 0.004, 3)
for K in strikes_sp:
    cap = cop.price("Caplet",   K)
    flo = cop.price("Floorlet", K)
    print(f"  Strike={100*K:.2f}%  |  Caplet: {round(1e4*cap,2):6.2f} bps  |  Floorlet: {round(1e4*flo,2):6.2f} bps")


---
## IV. American Options Pricing

Three independent methods are implemented and benchmarked:

| Method | Complexity | Accuracy | Notes |
|---|---|---|---|
| **Binomial Tree** | $O(N^2)$ space | High for large $N$ | Reference; $N=10^4$ steps |
| **BAW Approximation** | $O(1)$ per call | Good for short/long $T$ | Closed-form + root-finding for $S^*$ |
| **Longstaff-Schwartz LSMC** | $O(M \times N)$ | Stochastic convergence | Degree-19 polynomial regression |

The Binomial tree serves as the benchmark. BAW is the production-speed method.
LSMC is the most general (extendable to path-dependent payoffs).


### IV.1 — Binomial Tree (Cox-Ross-Rubinstein)

The CRR tree uses log-symmetric up/down moves $u = e^{\sigma\sqrt{\Delta t}}$, $d = 1/u$ and
risk-neutral probability $p = \frac{e^{(r-q)\Delta t} - d}{u - d}$.

Backward induction with early-exercise test at each node:

$$V_t(S_t) = \max\!\left(\phi(S_t - K),\; e^{-r\Delta t}\left[p\,V_{t+dt}^u + (1-p)\,V_{t+dt}^d\right]\right)$$


In [ ]:
class binomial_model:
    """
    Cox-Ross-Rubinstein binomial tree for European and American options.
    Memory: O(N) via in-place backward pass on a 1D payoff vector.
    """
    def __init__(self, n_steps=10_000):
        self.n_steps  = n_steps
        self._has_data = False

    def set_option_data(self, S0, T, vol, r, q=0.0):
        """
        S0  : spot price
        T   : maturity (years)
        vol : Black-Scholes implied volatility (flat smile assumed)
        r   : continuously-compounded risk-free rate
        q   : continuous dividend yield
        """
        self.S0, self.T, self.vol, self.r, self.q = S0, T, vol, r, q
        self._has_data = True

    def binomial_price(self, isCall, K, amer=True):
        """
        Price a European or American option.

        Parameters
        ----------
        isCall : bool  — True = call, False = put
        K      : float — strike
        amer   : bool  — True = American, False = European
        """
        if not self._has_data:
            print("Please set option data first.")
            return None

        dt       = self.T / self.n_steps
        u        = math.exp(self.vol * math.sqrt(dt))
        d        = 1.0 / u
        p        = (math.exp((self.r - self.q) * dt) - d) / (u - d)
        discount = math.exp(-self.r * dt)
        phi      = 1 if isCall else -1

        # Terminal spot prices: S0 * u^(n - 2i) for i = 0,...,n_steps
        j = np.arange(self.n_steps + 1)
        S_T = self.S0 * u**(self.n_steps - 2 * j)

        # Option payoff at maturity
        v = np.maximum(phi * (S_T - K), 0.0)

        # Backward induction (in-place; only need spot prices at each node level)
        S_t = S_T.copy()
        for i in range(self.n_steps - 1, -1, -1):
            S_t = S_t[:i + 1] * u**(np.arange(i, -1, -1) * 2 - i)  # recompute
            # Continuation value
            v = discount * (p * v[:i + 1] + (1 - p) * v[1:i + 2])
            if amer:
                # Spot at each node at time step i
                S_node = self.S0 * u**(i - 2 * np.arange(i + 1))
                v = np.maximum(phi * (S_node - K), v)

        return v[0]


### IV.2 — Barone-Adesi & Whaley Analytical Approximation

The BAW model decomposes the American price as:

$$V^{\text{Amer}} = V^{\text{BS-Eur}} + A \cdot \left(\frac{S}{S^*}\right)^b, \quad \phi(S - S^*) < 0$$
$$V^{\text{Amer}} = \phi(S - K), \quad \phi(S - S^*) \ge 0$$

where $S^*$ (exercise frontier) solves a nonlinear equation via Newton-Raphson / bisection hybrid.


In [ ]:
class baw_model:
    """
    Barone-Adesi & Whaley analytical approximation for American options.
    Uses a Newton-Raphson / bisection hybrid to locate the optimal exercise boundary S*.
    """
    def __init__(self):
        self._has_data = False

    def set_option_data(self, S0, T, vol, r, q=0.0):
        self.S0, self.T, self.vol, self.r, self.q = S0, T, vol, r, q
        self._has_data = True

    # ── Private helpers ────────────────────────────────────────────────────────
    def _d1(self, S, K):
        """Black-Scholes d1."""
        v2T = self.vol**2 * self.T
        return (math.log(S / K) + (self.r - self.q) * self.T + v2T / 2) / v2T**0.5

    def _bs_price(self, isCall, S, K):
        """Black-Scholes present value."""
        phi = 1 if isCall else -1
        d1  = self._d1(S, K)
        d2  = d1 - self.vol * math.sqrt(self.T)
        return phi * (S * math.exp(-self.q * self.T) * norm.cdf(phi * d1)
                      - K * math.exp(-self.r * self.T) * norm.cdf(phi * d2))

    def _abs_delta(self, isCall, S, K):
        """Absolute Black-Scholes delta."""
        phi = 1 if isCall else -1
        return math.exp(-self.q * self.T) * norm.cdf(phi * self._d1(S, K))

    def _b_exponent(self, isCall):
        """BAW quadratic-approximation exponent b."""
        phi = 1 if isCall else -1
        M   = 2 * self.r / self.vol**2
        N   = 2 * (self.r - self.q) / self.vol**2
        a_  = 1 - math.exp(-self.r * self.T)
        return 0.5 * (1 - N + phi * math.sqrt((1 - N)**2 + 4 * M / a_))

    def _american_adj(self, isCall, S, S_star, K):
        """Early-exercise premium A*(S/S*)^b."""
        phi = 1 if isCall else -1
        b_  = self._b_exponent(isCall)
        A   = phi * (S_star / b_) * (1 - self._abs_delta(isCall, S_star, K))
        return A * (S / S_star)**b_

    def _obj_func(self, isCall, S, K):
        """Equation whose root gives S*: BS_price + adj - intrinsic = 0."""
        phi = 1 if isCall else -1
        return (self._bs_price(isCall, S, K)
                + self._american_adj(isCall, S, S, K)
                - phi * (S - K))

    def _obj_func_deriv(self, isCall, S, K):
        """Derivative of the objective function (for Newton-Raphson)."""
        phi = 1 if isCall else -1
        b_  = self._b_exponent(isCall)
        delta = phi * self._abs_delta(isCall, S, K)
        return (delta * (1 - 1 / b_)
                + phi * (1 / b_) * (1 - phi * math.exp(-self.q * self.T)
                * norm.pdf(phi * self._d1(S, K)) / (self.vol * math.sqrt(self.T)))
                - phi)

    # ── Optimal exercise boundary ──────────────────────────────────────────────
    def optimal_spot(self, isCall, K, n_max=750, n_stdev=3):
        """
        Locate S* via Newton-Raphson (fast) or bisection (fallback).
        Search range: [Fwd / exp(n_stdev*sigma*sqrt(T)), Fwd * exp(n_stdev*sigma*sqrt(T))].
        """
        Fwd   = self.S0 * math.exp((self.r - self.q) * self.T)
        S_min = Fwd / math.exp(n_stdev * self.vol * math.sqrt(self.T))
        S_max = Fwd * math.exp(n_stdev * self.vol * math.sqrt(self.T))

        # Newton-Raphson if the root is not bracketed
        if self._obj_func(isCall, S_min, K) * self._obj_func(isCall, S_max, K) > 0:
            return optimize.newton(
                lambda S: self._obj_func(isCall, S, K),
                self.S0,
                lambda S: self._obj_func_deriv(isCall, S, K),
                maxiter=n_max
            )

        # Bisection
        for _ in range(n_max):
            S_mid = (S_min + S_max) / 2
            f_mid = self._obj_func(isCall, S_mid, K)
            if f_mid == 0 or abs(S_max - S_min) < 1e-6:
                return S_mid
            if f_mid < 0:
                S_min = S_mid
            else:
                S_max = S_mid
        raise RuntimeError(f"BAW root-finding failed to converge at K={K}.")

    # ── BAW price ──────────────────────────────────────────────────────────────
    def baw_price(self, isCall, K):
        """
        Barone-Adesi & Whaley American option price.
        Early exercise of a call on a non-dividend-paying stock is never optimal -> return BS price.
        """
        if not self._has_data:
            print("Please set option data first.")
            return None

        if isCall and self.r > 0 and self.q == 0:
            return self._bs_price(isCall, self.S0, K)

        phi     = 1 if isCall else -1
        S_star  = self.optimal_spot(isCall, K)
        # Early exercise check
        if phi * (self.S0 - S_star) >= 0:
            return phi * (self.S0 - K)
        return self._bs_price(isCall, self.S0, K) + self._american_adj(isCall, self.S0, S_star, K)


### IV.3 — Longstaff-Schwartz Least-Squares Monte Carlo

Spots are simulated under the risk-neutral GBM:
$$S_{t+dt} = S_t \exp\!\left[\left(r - q - \tfrac{\sigma^2}{2}\right)dt + \sigma\sqrt{dt}\,Z_t\right]$$

Backward induction uses a degree-$p$ polynomial regression of discounted future option values
onto current spot to estimate the continuation value $C_t(S_t)$. Early exercise occurs when
intrinsic value $\geq C_t$.


In [ ]:
class ls_mc:
    """
    Longstaff-Schwartz Least-Squares Monte Carlo for American options.

    Parameters
    ----------
    n_simul   : number of Monte Carlo paths
    n_steps   : number of time steps per path
    reg_order : degree of the polynomial regression (Longstaff-Schwartz basis)
    """
    def __init__(self, n_simul=600_000, n_steps=100, reg_order=19):
        self.n_simul   = n_simul
        self.n_steps   = n_steps
        self.reg_order = reg_order
        self._has_data = False

    def set_option_data(self, S0, T, vol, r, q=0.0):
        self.S0, self.T, self.vol, self.r, self.q = S0, T, vol, r, q
        self._has_data = True

    def set_path(self):
        """
        Simulate n_simul GBM paths with n_steps time steps.
        Uses antithetic variates via the full normal matrix (no explicit antithetics here;
        variance reduction is achieved by the large n_simul).
        """
        if not self._has_data:
            print("Please set option data first.")
            return

        self.dt  = self.T / self.n_steps
        mu       = self.r - self.q
        sqrt_dt  = math.sqrt(self.dt)

        Z = np.random.normal(0, 1, (self.n_steps, self.n_simul))
        self.S = np.empty((self.n_steps + 1, self.n_simul))
        self.S[0] = self.S0

        for j in range(self.n_steps):
            self.S[j + 1] = self.S[j] * np.exp(
                (mu - 0.5 * self.vol**2) * self.dt + self.vol * sqrt_dt * Z[j]
            )

    def mc_price(self, isCall, K):
        """
        American option price via Longstaff-Schwartz.

        Backward pass: at each time step t, regress df*V_{t+dt} on S_t^0,...,S_t^p
        to estimate the continuation value, then apply the early-exercise decision.
        """
        if not self._has_data:
            print("Please set option data first.")
            return None
        if not hasattr(self, 'S'):
            print("Please run set_path() first.")
            return None

        phi = 1 if isCall else -1
        df  = math.exp(-self.r * self.dt)

        # Intrinsic value along all paths and all time steps
        exercise = np.maximum(phi * (self.S - K), 0.0)

        # Initialise with terminal payoff
        V = np.empty((self.n_steps + 1, self.n_simul))
        V[-1] = exercise[-1]

        for j in range(self.n_steps - 1, 0, -1):
            # Polynomial regression of discounted future value on current spot
            coeff = np.polyfit(self.S[j], df * V[j + 1], self.reg_order)
            C     = np.polyval(coeff, self.S[j])   # continuation value
            # Early exercise: exercise if intrinsic >= continuation
            V[j] = np.where(exercise[j] >= C, exercise[j], df * V[j + 1])

        V[0] = df * V[1]
        # Floor at intrinsic value of spot (prevents negative prices due to regression noise)
        return max(np.mean(V[0]), phi * (self.S0 - K))


### IV.4 — Numerical Application and Method Comparison

In [ ]:
# ── Contract parameters ──────────────────────────────────────────────────────
N_size = 100.0   # contract multiplier
S0_eq  = 120.0   # spot
T_eq   = 0.5     # maturity (years)
vol_eq = 0.35    # implied vol
r_eq   = 0.03    # risk-free rate
q_eq   = 0.01    # dividend yield
F_eq   = S0_eq * math.exp((r_eq - q_eq) * T_eq)

print(f"Contract Size: {N_size}  |  S0={S0_eq}  |  T={T_eq}  |  vol={vol_eq}  |  r={r_eq}  |  q={q_eq}")
print(f"Forward: {round(F_eq, 4)}")

# ── Model initialisation ─────────────────────────────────────────────────────
binom = binomial_model()
binom.set_option_data(S0_eq, T_eq, vol_eq, r_eq, q_eq)

baw = baw_model()
baw.set_option_data(S0_eq, T_eq, vol_eq, r_eq, q_eq)

mc = ls_mc()
mc.set_option_data(S0_eq, T_eq, vol_eq, r_eq, q_eq)
mc.set_path()


In [ ]:
# ── Pricing comparison ────────────────────────────────────────────────────────
scenarios = [
    ("OTM Put  (K = 95% * Fwd)", False, 0.95 * F_eq),
    ("OTM Call (K = 105% * Fwd)", True, 1.05 * F_eq),
]

print(f"{'Scenario':<30} {'Binomial':>10} {'BAW':>10} {'LSMC':>10}")
print("─" * 65)
for label, isCall, K in scenarios:
    p_binom = round(N_size * binom.binomial_price(isCall, K), 2)
    p_baw   = round(N_size * baw.baw_price(isCall, K),       2)
    p_mc    = round(N_size * mc.mc_price(isCall, K),          2)
    print(f"{label:<30} {p_binom:>10.2f} {p_baw:>10.2f} {p_mc:>10.2f}")


In [ ]:
# ── Convergence of binomial price vs number of steps ──────────────────────────
K_bench = 0.95 * F_eq
steps   = [50, 100, 250, 500, 1000, 2500, 5000]
prices  = []
for n in steps:
    bm = binomial_model(n_steps=n)
    bm.set_option_data(S0_eq, T_eq, vol_eq, r_eq, q_eq)
    prices.append(N_size * bm.binomial_price(False, K_bench))

plt.figure(figsize=(9, 4))
plt.semilogx(steps, prices, marker='o', label="Binomial Price")
plt.axhline(N_size * baw.baw_price(False, K_bench), color='red', linestyle='--', label="BAW Reference")
plt.xlabel("Number of Steps (log scale)"); plt.ylabel("Option Price")
plt.title("Binomial Convergence vs BAW — OTM American Put")
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
